# Notebook 08: Tri-Modal Training

**Variant 5:** Three-way contrastive alignment across query, API documentation, and code signature.

L = L(query <> doc) + lambda * (L(query <> code) + L(doc <> code))

The code modality (typed function signatures) captures functional structure that text descriptions miss, helping disambiguate APIs like `get_current_weather` vs `get_weather_forecast`.

**Prerequisite:** Run `01_index_apis.ipynb` first.

In [ ]:
import os, sys, json, random
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))

from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_dfsdt_negatives
from models.embeddings import format_api_string, format_api_code

In [ ]:
corpus = load_api_corpus(TOOLBENCH_DIR / 'toolenv' / 'tools')
lookup = build_api_lookup(corpus)

TRAIN_PATH = TOOLBENCH_DIR / 'toolllama_G123_dfs_train.json'
assert TRAIN_PATH.exists(), f'Training file not found: {TRAIN_PATH}'

train_examples = load_eval_examples(TRAIN_PATH)
with open(TRAIN_PATH) as f:
    raw_train = json.load(f)

# Preview the code format
sample = corpus[0]
print(f'Doc:  {format_api_string(sample)}')
print(f'Code: {format_api_code(sample)}')
print(f'\nCorpus: {len(corpus)} APIs | Training: {len(train_examples)} examples')

## Tri-Modal Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TriModalLoss(nn.Module):
    def __init__(self, model, scale=20.0, lambda_code=0.5):
        super().__init__()
        self.model = model
        self.scale = scale
        self.lambda_code = lambda_code

    def forward(self, sentence_features, labels):
        # [0]=query, [1]=doc, [2]=code, [3:]=hard negatives (doc format)
        reps = [self.model(sf)['sentence_embedding'] for sf in sentence_features]
        queries = reps[0]
        docs = reps[1]
        codes = reps[2]
        target = torch.arange(queries.size(0), device=queries.device)

        # Query <> Doc (with hard negatives)
        doc_pool = torch.cat(reps[1:2] + reps[3:], dim=0)
        sim_qd = F.cosine_similarity(queries.unsqueeze(1), doc_pool.unsqueeze(0), dim=-1) * self.scale
        loss_qd = F.cross_entropy(sim_qd, target)

        # Query <> Code (in-batch negatives)
        sim_qc = F.cosine_similarity(queries.unsqueeze(1), codes.unsqueeze(0), dim=-1) * self.scale
        loss_qc = F.cross_entropy(sim_qc, target)

        # Doc <> Code (in-batch negatives)
        sim_dc = F.cosine_similarity(docs.unsqueeze(1), codes.unsqueeze(0), dim=-1) * self.scale
        loss_dc = F.cross_entropy(sim_dc, target)

        return loss_qd + self.lambda_code * (loss_qc + loss_dc)

## Build Training Data

Each example: [query, api_doc, api_code, hard_neg1, ..., hard_neg7]

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader

BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
EPOCHS = 5
BATCH_SIZE = 64
N_HARD_NEGATIVES = 7
LAMBDA_CODE = 0.5

pairs = []
for ex in train_examples:
    raw_ex = raw_train[ex['raw_idx']]
    for name in ex['ground_truth_apis']:
        if name not in lookup:
            continue
        api = lookup[name]
        hard_negs = build_dfsdt_negatives(raw_ex, corpus, ex['ground_truth_apis'], lookup, n=N_HARD_NEGATIVES)
        neg_texts = [format_api_string(n) for n in hard_negs]
        if len(neg_texts) < N_HARD_NEGATIVES:
            continue
        pairs.append(InputExample(texts=[
            ex['user_query'],
            format_api_string(api),
            format_api_code(api),
        ] + neg_texts))

random.shuffle(pairs)
print(f'Variant 5: {len(pairs)} training pairs')

## Train

In [ ]:
model = SentenceTransformer(BASE_MODEL)
loader = DataLoader(pairs, shuffle=True, batch_size=BATCH_SIZE)
loss = TriModalLoss(model, lambda_code=LAMBDA_CODE)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(loader) * EPOCHS),
    output_path=str(PROJECT_DIR / 'checkpoints' / 'v5_trimodal'),
    show_progress_bar=True,
)
print('Variant 5 saved to checkpoints/v5_trimodal')